In [1]:
import pandas as pd, numpy as np, plotly.express as px
print(pd.__version__)

3.1.1


# E-Commerce Sales Analysis (Olist)

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

pd.set_option("display.max_columns", None)
DATA = Path("../data")

In [3]:
files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category": "product_category_name_translation.csv",
}

dfs = {name: pd.read_csv(DATA / f) for name, f in files.items()}

for name, df in dfs.items():
    print(f"{name:12} {df.shape[0]:>8} lignes  {df.shape[1]:>2} colonnes")

customers       99441 lignes   5 colonnes
geolocation   1000163 lignes   5 colonnes
items          112650 lignes   7 colonnes
payments       103886 lignes   5 colonnes
reviews         99224 lignes   7 colonnes
orders          99441 lignes   8 colonnes
products        32951 lignes   9 colonnes
sellers          3095 lignes   4 colonnes
category           71 lignes   2 colonnes


In [4]:
for name, df in dfs.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]
    print(f"\n--- {name} ---")
    print(missing if len(missing) else "Aucune valeur manquante")


--- customers ---
Aucune valeur manquante

--- geolocation ---
Aucune valeur manquante

--- items ---
Aucune valeur manquante

--- payments ---
Aucune valeur manquante

--- reviews ---
review_comment_title      87656
review_comment_message    58247
dtype: int64

--- orders ---
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

--- products ---
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

--- sellers ---
Aucune valeur manquante

--- category ---
Aucune valeur manquante


In [5]:
orders = dfs["orders"].copy()
print(orders["order_status"].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [6]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c])

orders = orders[orders["order_status"] == "delivered"].copy()
orders = orders.dropna(subset=["order_delivered_customer_date"])

print(orders.shape)
print(orders["order_purchase_timestamp"].min(), "->", orders["order_purchase_timestamp"].max())

(96470, 8)
2016-09-15 12:16:38 -> 2018-08-29 15:00:37


In [7]:
products = dfs["products"].merge(dfs["category"], on="product_category_name", how="left")
products["category"] = products["product_category_name_english"].fillna("unknown")
products = products[["product_id", "category"]]

reviews = dfs["reviews"].groupby("order_id", as_index=False)["review_score"].mean()

customers = dfs["customers"][["customer_id", "customer_unique_id", "customer_city", "customer_state"]]

In [8]:
df = (
    orders
    .merge(dfs["items"], on="order_id", how="inner")
    .merge(products, on="product_id", how="left")
    .merge(customers, on="customer_id", how="left")
    .merge(reviews, on="order_id", how="left")
)

df["revenue"] = df["price"]
df["month"] = df["order_purchase_timestamp"].dt.to_period("M").astype(str)
df["delivery_days"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.days
df["is_late"] = df["order_delivered_customer_date"] > df["order_estimated_delivery_date"]

print(df.shape)
df.head()

(110189, 23)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,category,customer_unique_id,customer_city,customer_state,review_score,revenue,month,delivery_days,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,housewares,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,4.0,29.99,2017-10,8,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumery,af07308b275d755c9edb36a90c618231,barreiras,BA,4.0,118.70,2018-07,13,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,auto,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,5.0,159.90,2018-08,9,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,5.0,45.00,2017-11,13,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,stationery,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,5.0,19.90,2018-02,2,False


In [9]:
print("Doublons :", df.duplicated(subset=["order_id", "order_item_id"]).sum())
print("Revenu total :", f"{df['revenue'].sum():,.0f}")
print("Commandes uniques :", df["order_id"].nunique())
print(df[["category", "customer_state", "review_score", "delivery_days"]].isna().sum())

Doublons : 0
Revenu total : 13,220,249
Commandes uniques : 96470
category            0
customer_state      0
review_score      827
delivery_days       0
dtype: int64


In [10]:
df = df[(df["order_purchase_timestamp"] >= "2017-01-01") &
        (df["order_purchase_timestamp"] < "2018-09-01")].copy()
print(df.shape)

(109872, 23)


In [11]:
total_revenue = df["revenue"].sum()
n_orders = df["order_id"].nunique()
n_customers = df["customer_unique_id"].nunique()
avg_basket = df.groupby("order_id")["revenue"].sum().mean()
repeat_rate = (df.groupby("customer_unique_id")["order_id"].nunique() > 1).mean()

print(f"Revenu total      : {total_revenue:,.0f} BRL")
print(f"Commandes         : {n_orders:,}")
print(f"Clients uniques   : {n_customers:,}")
print(f"Panier moyen      : {avg_basket:,.2f} BRL")
print(f"Clients récurrents: {repeat_rate:.1%}")

Revenu total      : 13,179,778 BRL
Commandes         : 96,203
Clients uniques   : 93,096
Panier moyen      : 137.00 BRL
Clients récurrents: 3.0%


In [12]:
monthly = df.groupby("month", as_index=False).agg(
    revenue=("revenue", "sum"),
    orders=("order_id", "nunique"),
)
fig = px.line(monthly, x="month", y="revenue", markers=True,
              title="Revenu mensuel (BRL)")
fig.show()

Le revenue a ete multiplie par 4 en Janvier 2017 et April 2018

In [13]:
m = monthly.set_index("month")["revenue"]
print(m.round(0))
print("\nPic :", m.idxmax(), f"{m.max():,.0f}")
print("Avril 2018 / Janvier 2017 :", round(m["2018-04"] / m["2017-01"], 1))
print("Novembre 2017 / Octobre 2017 :", f"{m['2017-11'] / m['2017-10'] - 1:+.0%}")
print("Décembre 2017 / Novembre 2017 :", f"{m['2017-12'] / m['2017-11'] - 1:+.0%}")

month
2017-01    111798.0
2017-02    234223.0
2017-03    359199.0
2017-04    340670.0
2017-05    489159.0
2017-06    421923.0
2017-07    481605.0
2017-08    554700.0
2017-09    607400.0
2017-10    648248.0
2017-11    987648.0
2017-12    726033.0
2018-01    924645.0
2018-02    826437.0
2018-03    953356.0
2018-04    973534.0
2018-05    977545.0
2018-06    855592.0
2018-07    867486.0
2018-08    838577.0
Name: revenue, dtype: float64

Pic : 2017-11 987,648
Avril 2018 / Janvier 2017 : 8.7
Novembre 2017 / Octobre 2017 : +52%
Décembre 2017 / Novembre 2017 : -26%


In [14]:
monthly = df.groupby("month", as_index=False).agg(
    revenue=("revenue", "sum"),
    orders=("order_id", "nunique"),
)
fig = px.line(monthly, x="month", y="revenue", markers=True,
              title="Revenu mensuel (BRL)")
fig.show()

Le revenu mensuel est passé de 111 798 BRL en janvier 2017 à 973 534 BRL en avril 2018 (×8,7)

In [16]:
top_cat = (df.groupby("category", as_index=False)["revenue"].sum()
             .sort_values("revenue", ascending=False).head(10))
fig = px.bar(top_cat, x="revenue", y="category", orientation="h",
             title="Top 10 catégories par revenu")
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

In [17]:
by_state = (df.groupby("customer_state", as_index=False)["revenue"].sum()
              .sort_values("revenue", ascending=False))
fig = px.bar(by_state, x="customer_state", y="revenue",
             title="Revenu par État brésilien")
fig.show()

In [18]:
rev = df.dropna(subset=["review_score"]).drop_duplicates("order_id")
late = rev.groupby("is_late", as_index=False)["review_score"].mean()
late["is_late"] = late["is_late"].map({False: "À l'heure", True: "En retard"})

fig = px.bar(late, x="is_late", y="review_score", text_auto=".2f",
             title="Note moyenne selon le retard de livraison")
fig.update_yaxes(range=[0, 5])
fig.show()
print("Part des commandes en retard :", f"{rev['is_late'].mean():.1%}")

Part des commandes en retard : 8.0%


In [19]:
rev["delay_days"] = (rev["order_delivered_customer_date"] - rev["order_estimated_delivery_date"]).dt.days
rev["delay_bucket"] = pd.cut(rev["delay_days"], bins=[-999, 0, 3, 7, 999],
                             labels=["À l'heure", "1-3 j", "4-7 j", "8+ j"])
px.bar(rev.groupby("delay_bucket", as_index=False)["review_score"].mean(),
       x="delay_bucket", y="review_score", text_auto=".2f")